# Layer-wise Linear Probing Ablation

Studio ablativo: addestramento di un classificatore lineare sulle feature estratte
dall'uscita di ogni layer del ViT **UNI pre-addestrato** (nessun fine-tuning del backbone).

**Setup:**
- Backbone: UNI (ViT-L, 24 layer, 1024-dim) — **pesi congelati**
- Classificatore: singolo Linear layer + LayerNorm (addestrato da zero)
- Input del classificatore: token CLS all'uscita del layer x, con x ∈ {1, 2, ..., 24}
- Layer 24 = output finale (global pool, come usato normalmente)

**Obiettivo:** capire quanta informazione discriminativa si accumula layer per layer nel backbone pre-addestrato.

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import types, sys
import timm
from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
import torchvision.transforms as T

sys.path.append(".")
from src.dataset import HistologicalImageDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [2]:
DATASET_NAME = "BREAKHIS"   # cambia in "NCT-CRC-HE" per l'altro dataset

CFG = dict(
    data_dir     = f"/data/{DATASET_NAME}",
    img_size     = 224,
    batch_size   = 64,
    num_workers  = 4,
    seed         = 42,
    results_dir  = Path(f"results/{DATASET_NAME}/linear_probing_ablation"),
    # Layer da testare: 1..24 (24 = output finale del backbone)
    probe_layers = list(range(1, 25)),
    # Training del classificatore lineare
    lr           = 1e-3,
    weight_decay = 1e-4,
    epochs       = 20,
    far_threshold= 1e-4,
)

torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
CFG["results_dir"].mkdir(parents=True, exist_ok=True)
print(f"Layer da testare: {CFG['probe_layers']}")
print(f"Totale run: {len(CFG['probe_layers'])}")

Layer da testare: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
Totale run: 24


In [3]:
train_tf = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(90),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

eval_tf = T.Compose([
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

train_ds = HistologicalImageDataset(f"{CFG['data_dir']}/train", transform=train_tf)
val_ds   = HistologicalImageDataset(f"{CFG['data_dir']}/val",   transform=eval_tf)
test_ds  = HistologicalImageDataset(f"{CFG['data_dir']}/test",  transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], pin_memory=True,
                          persistent_workers=True)
val_loader   = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True,
                          persistent_workers=True)
test_loader  = DataLoader(test_ds, batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True,
                          persistent_workers=True)

CLASS_NAMES = test_ds.class_names
N_CLASSES   = len(CLASS_NAMES)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print(f"Classi ({N_CLASSES}): {CLASS_NAMES}")

Loading from /data/BREAKHIS/train...


Loading dataset from disk:   0%|          | 0/29 [00:00<?, ?it/s]

Loaded 25880 samples, 8 classes
Class distribution:
  adenosis: 1139 (4.4%)
  fibroadenoma: 3685 (14.2%)
  phyllodes_tumor: 1419 (5.5%)
  tubular_adenoma: 1642 (6.3%)
  ductal_carcinoma: 11717 (45.3%)
  lobular_carcinoma: 1927 (7.4%)
  mucinous_carcinoma: 2446 (9.5%)
  papillary_carcinoma: 1905 (7.4%)
Loading from /data/BREAKHIS/val...
Loaded 6832 samples, 8 classes
Class distribution:
  adenosis: 541 (7.9%)
  fibroadenoma: 692 (10.1%)
  phyllodes_tumor: 423 (6.2%)
  tubular_adenoma: 601 (8.8%)
  ductal_carcinoma: 2769 (40.5%)
  lobular_carcinoma: 601 (8.8%)
  mucinous_carcinoma: 757 (11.1%)
  papillary_carcinoma: 448 (6.6%)
Loading from /data/BREAKHIS/test...
Loaded 6833 samples, 8 classes
Class distribution:
  adenosis: 540 (7.9%)
  fibroadenoma: 693 (10.1%)
  phyllodes_tumor: 423 (6.2%)
  tubular_adenoma: 602 (8.8%)
  ductal_carcinoma: 2769 (40.5%)
  lobular_carcinoma: 602 (8.8%)
  mucinous_carcinoma: 757 (11.1%)
  papillary_carcinoma: 447 (6.5%)
Train: 25880 | Val: 6832 | Test: 683

In [4]:
# Carica UNI pre-addestrato — pesi completamente congelati
backbone = timm.create_model(
    "hf-hub:MahmoodLab/uni",
    pretrained=True,
    init_values=1e-5,
    dynamic_img_size=True,
).to(device)

for p in backbone.parameters():
    p.requires_grad_(False)
backbone.eval()

EMBED_DIM = backbone.embed_dim   # 1024
N_BLOCKS  = len(backbone.blocks) # 24
print(f"Backbone: embed_dim={EMBED_DIM}, n_blocks={N_BLOCKS}")

Backbone: embed_dim=1024, n_blocks=24


In [5]:
@torch.no_grad()
def extract_cls_at_layer(backbone, loader, target_layer: int, device):
    """
    Estrae il token CLS dopo il block `target_layer` (1-indexed).
    target_layer == N_BLOCKS usa l'output finale del backbone (global_pool).

    Ritorna:
        features: (N_samples, EMBED_DIM) float32
        labels:   (N_samples,) int64
    """
    block_idx = target_layer - 1   # 0-indexed

    all_feats, all_labels = [], []
    backbone.eval()

    for imgs, labels in tqdm(loader, leave=False, desc=f"Layer {target_layer}"):
        imgs = imgs.to(device, non_blocking=True)

        if target_layer == N_BLOCKS:
            # Output finale: usa il forward completo del backbone
            feats = backbone(imgs)   # (B, EMBED_DIM) grazie a global_pool
        else:
            # Uscita intermedia: forward parziale con hook
            captured = {}

            def hook_fn(module, input, output):
                # output: (B, N_tokens, EMBED_DIM)
                captured["cls"] = output[:, 0, :].float()  # solo CLS

            handle = backbone.blocks[block_idx].register_forward_hook(hook_fn)
            try:
                # Esegui il forward completo — il hook cattura l'uscita del block
                # voluto; i block successivi non interessano ma girano comunque.
                # Per evitare spreco computazionale, facciamo un forward parziale
                # eseguendo solo i primi `target_layer` block.
                x = backbone.patch_embed(imgs)
                x = backbone._pos_embed(x)
                x = backbone.patch_drop(x)
                x = backbone.norm_pre(x)
                for blk in backbone.blocks[:target_layer]:
                    x = blk(x)
                feats = x[:, 0, :].float()   # CLS token
            finally:
                handle.remove()

        all_feats.append(feats.cpu())
        all_labels.append(labels)

    return torch.cat(all_feats), torch.cat(all_labels)

In [6]:
def compute_tar_at_far(scores, is_correct, far_threshold=1e-4):
    scores    = np.array(scores)
    correct   = np.array(is_correct).astype(bool)
    incorrect = ~correct
    if incorrect.sum() == 0:
        return 1.0
    n_far     = max(1, int(np.ceil(incorrect.sum() * far_threshold)))
    threshold = np.sort(scores[incorrect])[::-1][min(n_far - 1, incorrect.sum() - 1)]
    return float((scores[correct] >= threshold).mean())


def compute_metrics(preds, labels, scores, far_threshold=1e-4):
    preds  = np.array(preds)
    labels = np.array(labels)
    scores = np.array(scores)
    f1     = f1_score(labels, preds, average="macro", zero_division=0)
    acc    = float((preds == labels).mean())
    tar    = compute_tar_at_far(scores, preds == labels, far_threshold)
    return {"accuracy": acc, "f1_macro": f1, "tar_at_far": tar}

In [7]:
class LinearHead(nn.Module):
    """Classificatore lineare: LayerNorm → Linear."""
    def __init__(self, embed_dim: int, n_classes: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, n_classes),
        )

    def forward(self, x):
        return self.net(x)


def train_linear_head(
    train_feats: torch.Tensor,
    train_labels: torch.Tensor,
    val_feats: torch.Tensor,
    val_labels: torch.Tensor,
    n_classes: int,
    embed_dim: int,
    cfg: dict,
    layer_idx: int,
) -> LinearHead:
    """
    Addestra un LinearHead sulle feature pre-estratte.
    Ritorna il modello con i migliori pesi sulla validation.
    """
    head = LinearHead(embed_dim, n_classes).to(device)

    # Dataset in-memory (feature già estratte)
    from torch.utils.data import TensorDataset
    tr_ds = TensorDataset(train_feats, train_labels)
    va_ds = TensorDataset(val_feats, val_labels)
    tr_dl = DataLoader(tr_ds, batch_size=256, shuffle=True)
    va_dl = DataLoader(va_ds, batch_size=256, shuffle=False)

    optimizer = torch.optim.AdamW(
        head.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg["epochs"]
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_val_f1 = -1.0
    best_state  = None

    pbar = tqdm(range(cfg["epochs"]), desc=f"  Layer {layer_idx:2d} train", leave=False)
    for epoch in pbar:
        # --- Train ---
        head.train()
        for feats_b, labels_b in tr_dl:
            feats_b, labels_b = feats_b.to(device), labels_b.to(device)
            optimizer.zero_grad()
            loss = criterion(head(feats_b), labels_b)
            loss.backward()
            optimizer.step()
        scheduler.step()

        # --- Validation ---
        head.eval()
        all_p, all_l, all_s = [], [], []
        with torch.no_grad():
            for feats_b, labels_b in va_dl:
                feats_b = feats_b.to(device)
                probs   = head(feats_b).softmax(-1).cpu()
                all_p.extend(probs.argmax(-1).numpy())
                all_l.extend(labels_b.numpy())
                all_s.extend(probs.max(-1).values.numpy())

        val_f1 = f1_score(all_l, all_p, average="macro", zero_division=0)
        pbar.set_postfix(epoch=epoch + 1, val_f1=f"{val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = {k: v.clone() for k, v in head.state_dict().items()}

    head.load_state_dict(best_state)
    return head


@torch.no_grad()
def evaluate_head(head, feats, labels, cfg):
    head.eval()
    from torch.utils.data import TensorDataset
    ds = TensorDataset(feats, labels)
    dl = DataLoader(ds, batch_size=256, shuffle=False)

    all_p, all_l, all_s = [], [], []
    for feats_b, labels_b in dl:
        feats_b = feats_b.to(device)
        probs   = head(feats_b).softmax(-1).cpu()
        all_p.extend(probs.argmax(-1).numpy())
        all_l.extend(labels_b.numpy())
        all_s.extend(probs.max(-1).values.numpy())

    return compute_metrics(all_p, all_l, all_s, cfg["far_threshold"])

In [ ]:
# === Ciclo principale: per ogni layer, estrai feature, addestra head, valuta ===

results = []

for layer in tqdm(CFG["probe_layers"], desc="Probe layer"):
    # 1. Estrazione feature (backbone congelato)
    train_feats, train_labels = extract_cls_at_layer(backbone, train_loader, layer, device)
    val_feats,   val_labels   = extract_cls_at_layer(backbone, val_loader,   layer, device)
    test_feats,  test_labels  = extract_cls_at_layer(backbone, test_loader,  layer, device)

    # 2. Addestramento classificatore lineare
    head = train_linear_head(
        train_feats, train_labels,
        val_feats,   val_labels,
        N_CLASSES, EMBED_DIM, CFG, layer
    )

    # 3. Valutazione su test set
    test_metrics = evaluate_head(head, test_feats, test_labels, CFG)
    val_metrics  = evaluate_head(head, val_feats,  val_labels,  CFG)

    row = {
        "layer"       : layer,
        "val_f1"      : val_metrics["f1_macro"],
        "val_acc"     : val_metrics["accuracy"],
        "test_f1"     : test_metrics["f1_macro"],
        "test_acc"    : test_metrics["accuracy"],
        "test_tar"    : test_metrics["tar_at_far"],
    }
    results.append(row)
    print(
        f"Layer {layer:2d} | "
        f"val_F1={val_metrics['f1_macro']:.4f} | "
        f"test_F1={test_metrics['f1_macro']:.4f} | "
        f"test_ACC={test_metrics['accuracy']:.4f} | "
        f"TAR={test_metrics['tar_at_far']:.4f}"
    )

df = pd.DataFrame(results)
csv_path = CFG["results_dir"] / "linear_probing_ablation.csv"
df.to_csv(csv_path, index=False)
print(f"\nRisultati salvati in: {csv_path}")

Probe layer:   0%|          | 0/24 [00:00<?, ?it/s]

Layer 1:   0%|          | 0/405 [00:00<?, ?it/s]

Layer 1:   0%|          | 0/107 [00:00<?, ?it/s]

Layer 1:   0%|          | 0/107 [00:00<?, ?it/s]

  Layer  1 train:   0%|          | 0/20 [00:00<?, ?it/s]

...

Layer 3:   0%|          | 0/107 [00:00<?, ?it/s]

Layer 3:   0%|          | 0/107 [00:00<?, ?it/s]

  Layer  3 train:   0%|          | 0/20 [00:00<?, ?it/s]

Layer  3 | val_F1=0.4143 | test_F1=0.4131 | test_ACC=0.5668 | TAR=0.0041


Layer 4:   0%|          | 0/405 [00:00<?, ?it/s]

In [ ]:
# === Plot principale: F1 e Accuracy per layer ===

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), dpi=150)

plot_data = [
    ("test_f1",  "F1 Macro",  "#2196F3", "Test F1 Macro vs Layer"),
    ("test_acc", "Accuracy",  "#4CAF50", "Accuracy vs Layer"),
    ("test_tar", "TAR@FAR",   "#FF9800", "TAR@FAR vs Layer"),
]

for ax, (col, ylabel, color, title) in zip(axes, plot_data):
    ax.plot(df["layer"], df[col], marker="o", color=color, lw=2.0, ms=5)
    ax.fill_between(df["layer"], df[col].min(), df[col], alpha=0.12, color=color)

    # Evidenzia layer con valore massimo
    best_row = df.loc[df[col].idxmax()]
    ax.scatter([best_row["layer"]], [best_row[col]],
               color="red", zorder=5, s=80,
               label=f"Best: layer {int(best_row['layer'])} ({best_row[col]:.4f})")

    ax.set_xlabel("Layer", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xticks(CFG["probe_layers"])
    ax.tick_params(axis="x", labelsize=8)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

fig.suptitle(
    f"Linear Probing Ablation – {DATASET_NAME}\n"
    f"UNI backbone congelato, classificatore lineare per layer",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
fig.savefig(CFG["results_dir"] / "linear_probing_ablation.png", dpi=220, bbox_inches="tight")
fig.savefig(CFG["results_dir"] / "linear_probing_ablation.pdf", bbox_inches="tight")
plt.show()
print("Plot salvato.")

In [ ]:
# === Plot combinato F1 + TAR su un unico asse con asse Y duale ===

fig, ax1 = plt.subplots(figsize=(10, 5), dpi=150)
ax2 = ax1.twinx()

l1, = ax1.plot(df["layer"], df["test_f1"],  marker="o", color="#2196F3",
               lw=2.0, ms=5, label="F1 Macro")
l2, = ax1.plot(df["layer"], df["test_acc"], marker="s", color="#4CAF50",
               lw=1.5, ms=4, ls="--", label="Accuracy")
l3, = ax2.plot(df["layer"], df["test_tar"], marker="^", color="#FF9800",
               lw=1.5, ms=4, ls=":", label="TAR@FAR")

ax1.set_xlabel("Layer", fontsize=12)
ax1.set_ylabel("F1 Macro / Accuracy", fontsize=11, color="#333")
ax2.set_ylabel("TAR@FAR", fontsize=11, color="#FF9800")
ax2.tick_params(axis="y", labelcolor="#FF9800")

ax1.set_xticks(CFG["probe_layers"])
ax1.tick_params(axis="x", labelsize=9)
ax1.grid(alpha=0.25)

lines = [l1, l2, l3]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, fontsize=10, loc="lower right")

ax1.set_title(
    f"Linear Probing Ablation – {DATASET_NAME}",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
fig.savefig(CFG["results_dir"] / "linear_probing_combined.png", dpi=220, bbox_inches="tight")
fig.savefig(CFG["results_dir"] / "linear_probing_combined.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# === Riepilogo testuale ===

print(f"{'Layer':>6} | {'Val F1':>8} | {'Test F1':>8} | {'Accuracy':>9} | {'TAR@FAR':>9}")
print("-" * 55)
for _, row in df.iterrows():
    best_marker = " ◄" if row["test_f1"] == df["test_f1"].max() else ""
    print(
        f"{int(row['layer']):>6} | "
        f"{row['val_f1']:>8.4f} | "
        f"{row['test_f1']:>8.4f} | "
        f"{row['test_acc']:>9.4f} | "
        f"{row['test_tar']:>9.4f}{best_marker}"
    )

best = df.loc[df["test_f1"].idxmax()]
print(f"\nMiglior layer per F1: {int(best['layer'])} "
      f"(F1={best['test_f1']:.4f}, ACC={best['test_acc']:.4f}, TAR={best['test_tar']:.4f})")

display(df.style.background_gradient(
    subset=["val_f1", "test_f1", "test_acc", "test_tar"],
    cmap="Blues"
).format("{:.4f}", subset=["val_f1", "test_f1", "test_acc", "test_tar"]))